# 02 · Hansen: bosque base y pérdida anual de cobertura

**Qué hace.** Baja los mapas de Hansen Global Forest Change (Universidad de Maryland) sobre Colombia y, para
cada celda de 5 km de la grilla, calcula dos cosas:

1. **El bosque base:** las hectáreas que eran bosque al empezar el análisis. Es el denominador común de las
   tablas de deforestación, y lo usan los notebooks 03, 04 y 05.
2. **La pérdida anual de cobertura arbórea:** cuántas hectáreas de ese bosque perdieron su dosel cada año,
   por cualquier causa (tala, incendio, cosecha de plantaciones, daño natural).

**Cuándo correrlo.** Después de `01_carga_grilla_y_municipios`. Para actualizar, una vez al año, cuando sale
una versión nueva de Hansen: se cambia `HANSEN_VERSION` en la celda 1 y se vuelve a correr.

**Qué lee.** La tabla `GRILLA` y los mapas de Hansen, que se bajan del almacenamiento público de Google
(unos 2 GB, sin cuenta ni llave).

**Qué escribe**, en `DATASET_DEFORESTACION`:

| Tabla | Qué es |
|---|---|
| `BOSQUE_BASE` | Una fila por celda de la grilla: hectáreas de bosque al empezar el análisis |
| `PANEL_HANSEN` | Una fila por celda y año: hectáreas de pérdida de cobertura arbórea |

**Qué tienes que cambiar.** Nada: la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`).

**Cuánto tarda.** Unos 20 minutos: 10 de descarga y 10 de cálculo.

**Qué mide y qué no.** La pérdida de Hansen no es deforestación oficial: incluye plantaciones y causas
naturales, y sobre Colombia equivale a entre 1,5 y 2,5 veces la cifra del IDEAM. Sirve para comparar con
otros países, porque aplica el mismo algoritmo en todo el planeta. La deforestación oficial la arma el
notebook 04.

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Para actualizar --------------------------------------------------
# Versión de Hansen. La Universidad de Maryland publica una cada año, con
# un año más de pérdida (después de GFC-2025-v1.13 vendrá algo como
# GFC-2026-v1.14). Para ver cuál es la última:
# https://storage.googleapis.com/earthenginepartners-hansen/
HANSEN_VERSION = "GFC-2025-v1.13"

# Primer año de pérdida que se guarda en panel_hansen. El último lo da la
# versión: GFC-2025 trae pérdida hasta 2025.
ANIO_INICIO = 2020

# --- Definición de bosque del trabajo de grado ------------------------
# No se cambian: si se cambian, las tablas dejan de ser comparables con
# las del trabajo y entre sí.
UMBRAL_DOSEL = 30         # % mínimo de cobertura de dosel en el año 2000
ANIO_MASCARA = 2019       # lo perdido hasta este año ya no cuenta como bosque
BOSQUE_MINIMO_HA = 50     # celdas con menos bosque base quedan fuera de los paneles
RESOLUCION_M = 25         # tamaño del píxel al reproyectar, en metros
BLOQUE_CELDAS = 20        # se procesa en bloques de 20 x 20 celdas (100 x 100 km)

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

# Carpeta donde se bajan los archivos mientras corre el notebook.
# Vacío: una carpeta temporal. Si ya los tienes bajados, apúntala ahí
# y no se vuelven a descargar.
CARPETA_DESCARGAS = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet rasterio==1.4.3 pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "02_carga_hansen"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'panel_ideam' -> 'proyecto.dataset.PANEL_IDEAM'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA, PANEL_IDEAM...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None, procedencia=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "panel_ideam".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    procedencia    {"fuente": ..., ...}: si se da, deja una fila en la tabla
                   registro_cargas con qué se cargó, de dónde y cuándo.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

    if procedencia is not None:
        fila = pd.DataFrame([{
            "tabla": tabla.upper(),
            "filas": int(t.num_rows),
            "fuente": str(procedencia.get("fuente", "")),
            "parametros": json.dumps({k: v for k, v in procedencia.items() if k != "fuente"},
                                     ensure_ascii=False, default=str),
            "notebook": NOMBRE_NOTEBOOK,
            "cargado_en": datetime.now(timezone.utc),
        }])
        client.load_table_from_dataframe(
            fila, nombre_completo("registro_cargas", dataset),
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

## 4. La grilla

Se lee de BigQuery la grilla que armó el notebook 01. `mapa_celda` traduce la posición de una celda en la
grilla, `(ix, iy)`, a su número de fila en la tabla.

In [ ]:
LADO_CELDA_M = 5_000          # los de la grilla (notebook 01)
CRS_GRILLA = "EPSG:3116"
AREA_PX_HA = RESOLUCION_M ** 2 / 10_000

grilla = leer_tabla("grilla")
mapa_celda = {(int(a), int(b)): i for i, (a, b) in enumerate(zip(grilla["ix"], grilla["iy"]))}
print(f"Grilla: {len(grilla):,} celdas")

## 5. Descargar Hansen

Hansen viene partido en cuadros de 10 × 10 grados, nombrados por su esquina superior izquierda (`10N_080W`
cubre de 0° a 10° de latitud y de 80° a 70° oeste). A Colombia la cubren seis cuadros, y de cada uno se
bajan dos capas:

- `treecover2000`: porcentaje de cobertura de dosel en el año 2000.
- `lossyear`: año en que el píxel perdió la cobertura (1 = 2001, …, 25 = 2025; 0 = nunca).

Cada archivo se baja primero con otro nombre y solo se renombra si llegó completo, así que una descarga
interrumpida nunca queda pasando por buena. Los que ya están en la carpeta no se vuelven a bajar.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

URL_HANSEN = "https://storage.googleapis.com/earthenginepartners-hansen"
BBOX_COLOMBIA = (-79.2, -4.4, -66.7, 13.5)   # lon_min, lat_min, lon_max, lat_max, con margen
DIR_HANSEN = DIR_TRABAJO
DIR_HANSEN.mkdir(parents=True, exist_ok=True)


def cuadros_hansen(bbox):
    """Nombres de los cuadros de 10 x 10 grados que tocan el rectángulo."""
    lon_min, lat_min, lon_max, lat_max = bbox
    nombres = []
    for arriba in range(int((lat_max // 10 + 1) * 10), int(lat_min // 10 * 10), -10):
        for izquierda in range(int(lon_min // 10 * 10), int((lon_max // 10 + 1) * 10), 10):
            ns = "N" if arriba >= 0 else "S"
            ew = "W" if izquierda < 0 else "E"
            nombres.append(f"{abs(arriba):02d}{ns}_{abs(izquierda):03d}{ew}")
    return nombres


def descargar(url, destino, minimo_bytes=1 << 20, intentos=3):
    """Baja un archivo por partes, con reintentos. Si ya está, no hace nada."""
    if destino.exists() and destino.stat().st_size > minimo_bytes:
        return True
    temporal = destino.with_suffix(destino.suffix + ".parcial")
    for intento in range(1, intentos + 1):
        try:
            with requests.get(url, stream=True, timeout=300) as r:
                r.raise_for_status()
                with temporal.open("wb") as f:
                    for trozo in r.iter_content(1 << 20):
                        f.write(trozo)
            if temporal.stat().st_size <= minimo_bytes:
                raise RuntimeError("el archivo llegó demasiado pequeño")
            temporal.replace(destino)
            return True
        except Exception as e:
            print(f"  {destino.name}: intento {intento} de {intentos} falló ({str(e)[:120]})")
            temporal.unlink(missing_ok=True)
    return False


cuadros = cuadros_hansen(BBOX_COLOMBIA)
tareas = [(f"{URL_HANSEN}/{HANSEN_VERSION}/Hansen_{HANSEN_VERSION}_{capa}_{c}.tif",
           DIR_HANSEN / f"Hansen_{HANSEN_VERSION}_{capa}_{c}.tif")
          for c in cuadros for capa in ("treecover2000", "lossyear")]
print(f"Cuadros de Hansen sobre Colombia: {cuadros}")

fallidos = []
with ThreadPoolExecutor(max_workers=6) as pool:
    futuros = {pool.submit(descargar, url, destino): destino for url, destino in tareas}
    for k, futuro in enumerate(as_completed(futuros), 1):
        destino = futuros[futuro]
        ok = futuro.result()
        mb = destino.stat().st_size / 1e6 if destino.exists() else 0
        print(f"  [{k}/{len(tareas)}] {'listo' if ok else 'FALLÓ'} {destino.name} ({mb:.0f} MB)")
        if not ok:
            fallidos.append(destino.name)
if fallidos:
    raise RuntimeError(f"No se pudieron bajar: {fallidos}. Vuelve a correr esta celda.")

TIF_DOSEL = sorted(DIR_HANSEN.glob(f"Hansen_{HANSEN_VERSION}_treecover2000_*.tif"))
TIF_PERDIDA = sorted(DIR_HANSEN.glob(f"Hansen_{HANSEN_VERSION}_lossyear_*.tif"))

## 6. Cómo se reparte un ráster entre las celdas

Estas funciones son el corazón geométrico de los notebooks 02 y 04, y son idénticas en los dos. Si cada
tabla repartiera los píxeles a su manera, una diferencia de medio píxel en el borde de una celda haría que
dejaran de ser comparables.

1. **Bloques.** La grilla se parte en bloques de 20 × 20 celdas, alineados con el origen. Solo se usan los
   bloques que tienen alguna celda de la grilla.
2. **Ráster de destino.** Para cada bloque se define un ráster de 25 m en la proyección de la grilla, que
   encaja exacto con los bordes de las celdas: 20 celdas × 5.000 m / 25 m = 4.000 píxeles, sin residuo.
3. **Reproyección.** Cada capa se reproyecta sobre ese ráster con vecino más cercano, porque las capas son
   categorías (porcentaje de dosel, año de pérdida, clase de cambio) y promediar categorías no tiene
   sentido.
4. **Celda de cada píxel.** Como el ráster de destino está en la proyección de la grilla, la celda de cada
   píxel es una división: `floor(x / 5000)`. Los bloques no se traslapan, así que los conteos de distintos
   bloques se suman sin contar nada dos veces.

In [ ]:
import rasterio
from rasterio.transform import Affine
from rasterio.warp import Resampling, reproject


def bloques(grilla):
    """Bloques (ix0, iy0, ix1, iy1), con ix1 e iy1 excluidos, que cubren las celdas de la grilla."""
    paso = BLOQUE_CELDAS
    ocupados = set(zip(grilla["ix"] // paso, grilla["iy"] // paso))
    return sorted((bx * paso, by * paso, (bx + 1) * paso, (by + 1) * paso) for bx, by in ocupados)


def perfil_bloque(ix0, iy0, ix1, iy1):
    """Ráster de destino del bloque: transformación, proyección y tamaño."""
    x_min, x_max = ix0 * LADO_CELDA_M, ix1 * LADO_CELDA_M
    y_min, y_max = iy0 * LADO_CELDA_M, iy1 * LADO_CELDA_M
    ancho = int(round((x_max - x_min) / RESOLUCION_M))
    alto = int(round((y_max - y_min) / RESOLUCION_M))
    # Origen arriba a la izquierda; -RESOLUCION_M porque las filas bajan y la Y sube.
    transform = Affine(RESOLUCION_M, 0, x_min, 0, -RESOLUCION_M, y_max)
    return {"transform": transform, "crs": CRS_GRILLA, "height": alto, "width": ancho}


def indice_celdas_bloque(perfil, mapa_celda):
    """Para cada píxel del bloque, su fila en la tabla de la grilla (-1 si cae fuera de la grilla).

    Como el ráster no está rotado, la X de un píxel depende solo de su
    columna y la Y solo de su fila: basta resolver las celdas de una fila
    y una columna y combinarlas, en vez de recorrer 16 millones de píxeles.
    Se usa el CENTRO del píxel (+0,5) para decidir su celda.
    """
    tr = perfil["transform"]
    cols = np.arange(perfil["width"]) + 0.5
    filas = np.arange(perfil["height"]) + 0.5
    ix_col = np.floor((tr.c + tr.a * cols) / LADO_CELDA_M).astype(np.int64)
    iy_fila = np.floor((tr.f + tr.e * filas) / LADO_CELDA_M).astype(np.int64)
    ux, pos_x = np.unique(ix_col, return_inverse=True)
    uy, pos_y = np.unique(iy_fila, return_inverse=True)
    tabla = np.array([[mapa_celda.get((int(a), int(b)), -1) for a in ux] for b in uy], dtype=np.int32)
    return tabla[pos_y[:, None], pos_x[None, :]]


def reproyectar_sobre_bloque(rutas, perfil, dtype=np.uint8):
    """Reproyecta uno o varios rásteres sobre el bloque y los combina.

    Donde un archivo no tiene datos, reproject() deja 0; el máximo entre
    archivos se queda con el valor real del que sí cubre el píxel. Eso
    supone que 0 es "sin dato", y así es en Hansen (0 = sin pérdida) y en
    el IDEAM (las clases van de 1 a 5).
    """
    salida = np.zeros((perfil["height"], perfil["width"]), dtype=dtype)
    for ruta in rutas:
        with rasterio.open(ruta) as src:
            trozo = np.zeros_like(salida)
            reproject(source=rasterio.band(src, 1), destination=trozo,
                      dst_transform=perfil["transform"], dst_crs=perfil["crs"],
                      resampling=Resampling.nearest, src_nodata=None, dst_nodata=0)
        salida = np.maximum(salida, trozo)
    return salida


def contar_por_celda_y_clase(valores, idx, n_celdas, n_clases):
    """Matriz (celdas x clases) con el número de píxeles de cada clase en cada celda.

    Un solo np.bincount sobre el índice combinado celda * n_clases + clase,
    en vez de uno por clase. Se descartan los píxeles fuera de la grilla
    (idx = -1) y los de clase 0 (sin dato).
    """
    plano_idx = idx.ravel()
    plano_val = valores.ravel().astype(np.int64)
    ok = (plano_idx >= 0) & (plano_val > 0) & (plano_val < n_clases)
    conteo = np.bincount(plano_idx[ok] * n_clases + plano_val[ok], minlength=n_celdas * n_clases)
    return conteo[:n_celdas * n_clases].reshape(n_celdas, n_clases)

## 7. Bosque base y pérdida anual, celda por celda

En un solo recorrido por los bloques se calculan las dos cosas, con la misma definición de bosque:

- **Bosque base.** Un píxel es bosque si en 2000 tenía al menos 30 % de dosel **y** no lo había perdido
  antes de 2020 (`lossyear` de 1 a 19). Lo segundo evita contar como bosque algo que ya se había talado
  antes de que empiecen las alertas del notebook 03.
- **Pérdida anual.** Se cuenta la pérdida solo en píxeles que en 2000 eran bosque (dosel de al menos 30 %).
  Un cultivo o una zona urbana que cambia de dosel no entra.

Cada píxel de 25 m son 0,0625 ha.

In [ ]:
import time

N_CLASES_LOSSYEAR = 40        # lossyear va de 0 a 25; queda espacio para versiones futuras
n_celdas = len(grilla)
bosque_ha = np.zeros(n_celdas)
perdida_ha = np.zeros((n_celdas, N_CLASES_LOSSYEAR), dtype=np.float64)
lista_bloques = bloques(grilla)
print(f"Recorriendo {len(lista_bloques)} bloques de {BLOQUE_CELDAS} x {BLOQUE_CELDAS} celdas a {RESOLUCION_M} m...")

inicio = time.time()
for k, (ix0, iy0, ix1, iy1) in enumerate(lista_bloques, 1):
    perfil = perfil_bloque(ix0, iy0, ix1, iy1)
    dosel = reproyectar_sobre_bloque(TIF_DOSEL, perfil)
    perdida = reproyectar_sobre_bloque(TIF_PERDIDA, perfil)
    idx = indice_celdas_bloque(perfil, mapa_celda)

    # Bosque base: dosel suficiente en 2000 y sin pérdida hasta ANIO_MASCARA.
    bosque = (dosel >= UMBRAL_DOSEL) & ~((perdida > 0) & (perdida <= ANIO_MASCARA - 2000))
    plano = idx.ravel()
    valido = (plano >= 0) & bosque.ravel()
    bosque_ha += np.bincount(plano[valido], minlength=n_celdas).astype(np.float64)[:n_celdas] * AREA_PX_HA

    # Pérdida anual: solo sobre lo que era bosque en 2000.
    perdida_en_bosque = np.where(dosel >= UMBRAL_DOSEL, perdida, 0)
    perdida_ha += contar_por_celda_y_clase(perdida_en_bosque, idx, n_celdas, N_CLASES_LOSSYEAR) * AREA_PX_HA

    if k % 20 == 0 or k == len(lista_bloques):
        print(f"  [{k}/{len(lista_bloques)}] bosque: {bosque_ha.sum():,.0f} ha  ·  "
              f"pérdida: {perdida_ha.sum():,.0f} ha  ·  {time.time() - inicio:.0f} s")

## 8. Guardar el bosque base

Se guarda para **todas** las celdas de la grilla, también las que tienen poco bosque: cada panel aplica
después el mínimo de 50 ha. A escala nacional deben salir unos 77,7 millones de hectáreas.

In [ ]:
bosque_base = pd.DataFrame({"cell_id": grilla["cell_id"].values, "bosque_base_ha": bosque_ha})
print(f"Bosque base nacional: {bosque_base.bosque_base_ha.sum():,.0f} ha; "
      f"celdas con al menos {BOSQUE_MINIMO_HA} ha: {(bosque_base.bosque_base_ha >= BOSQUE_MINIMO_HA).sum():,}")

guardar_tabla(
    bosque_base, "bosque_base",
    f"Bosque al empezar el análisis, por celda de la grilla, según Hansen {HANSEN_VERSION}: píxeles con "
    f"dosel de al menos {UMBRAL_DOSEL} % en 2000 y sin pérdida hasta {ANIO_MASCARA}, reproyectados a "
    f"{RESOLUCION_M} m. Es el denominador común de los paneles de deforestación.",
    descripciones={
        "cell_id": "Identificador de la celda (tabla GRILLA).",
        "bosque_base_ha": f"Hectáreas de bosque al empezar {ANIO_MASCARA + 1}, según Hansen.",
    },
    procedencia={"fuente": f"Hansen Global Forest Change {HANSEN_VERSION}", "umbral_dosel": UMBRAL_DOSEL,
                 "anio_mascara": ANIO_MASCARA, "resolucion_m": RESOLUCION_M})

## 9. La tabla de pérdida anual

Una fila por celda y año, de `ANIO_INICIO` al último año de la versión. Se quedan solo las celdas con al
menos 50 ha de bosque base, el mismo filtro de los otros paneles: una celda de páramo o de sabana no puede
perder bosque y solo agrega ceros.

In [ ]:
ANIO_FIN = int(HANSEN_VERSION.split("-")[1])        # "GFC-2025-v1.13" -> 2025
filas = []
for anio in range(ANIO_INICIO, ANIO_FIN + 1):
    codigo = anio - 2000                            # así codifica el año la capa lossyear
    if 0 < codigo < N_CLASES_LOSSYEAR:
        filas.append(pd.DataFrame({"cell_id": grilla["cell_id"].values, "anio": anio,
                                   "perdida_ha": perdida_ha[:, codigo]}))
panel_hansen = pd.concat(filas, ignore_index=True)

meta = grilla[["cell_id", "lon", "lat", "departamento"]].copy()
meta["bosque_base_ha"] = bosque_ha
panel_hansen = panel_hansen.merge(meta, on="cell_id", how="left")
panel_hansen = panel_hansen[panel_hansen["bosque_base_ha"] >= BOSQUE_MINIMO_HA].copy()
panel_hansen = panel_hansen.merge(grilla[["cell_id", "cod_dane", "municipio"]], on="cell_id", how="left")
panel_hansen = panel_hansen.sort_values(["cell_id", "anio"]).reset_index(drop=True)

print(f"panel_hansen: {len(panel_hansen):,} filas = {panel_hansen.cell_id.nunique():,} celdas x "
      f"{panel_hansen.anio.nunique()} años")
print("Pérdida de cobertura arbórea por año (ha):")
print(panel_hansen.groupby("anio")["perdida_ha"].sum().round(0).to_string())

In [ ]:
guardar_tabla(
    panel_hansen, "panel_hansen",
    f"Pérdida anual de cobertura arbórea por celda de 5 km, según Hansen {HANSEN_VERSION}. Mide pérdida de "
    "dosel por cualquier causa, no deforestación oficial: sobre Colombia equivale a 1,5-2,5 veces la cifra "
    f"del IDEAM. Solo celdas con al menos {BOSQUE_MINIMO_HA} ha de bosque base.",
    descripciones={
        "cell_id": "Identificador de la celda (tabla GRILLA).",
        "anio": "Año de la pérdida (año calendario).",
        "perdida_ha": f"Hectáreas que perdieron el dosel ese año, entre las que tenían al menos "
                      f"{UMBRAL_DOSEL} % de dosel en 2000.",
        "lon": "Longitud del centro de la celda.",
        "lat": "Latitud del centro de la celda.",
        "departamento": "Departamento del centro de la celda.",
        "bosque_base_ha": "Bosque al empezar el análisis (tabla BOSQUE_BASE). Es el denominador de esta tabla.",
        "cod_dane": "Código DANE del municipio del centro de la celda.",
        "municipio": "Nombre del municipio, solo para leer.",
    },
    procedencia={"fuente": f"Hansen Global Forest Change {HANSEN_VERSION}",
                 "anios": f"{ANIO_INICIO}-{ANIO_FIN}", "bosque_minimo_ha": BOSQUE_MINIMO_HA})

Listo. Sigue `03_carga_alertas_gfw`, que usa el bosque base.